In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

schema = StructType([
    StructField("event_time", StringType(), True),
    StructField("status", StringType(), True)
])

# Create data
data = [
    ("10:01", "on"), ("10:02", "on"), ("10:03", "on"), ("10:04", "off"),
    ("10:05", "on"), ("10:06", "off"), ("10:07", "on"), ("10:08", "on"),
    ("10:09", "off"), ("10:11", "on"), ("10:12", "off")
]

# Create DataFrame
event_status = spark.createDataFrame(data, schema)

# Show DataFrame
event_status.show()

+----------+------+
|event_time|status|
+----------+------+
|     10:01|    on|
|     10:02|    on|
|     10:03|    on|
|     10:04|   off|
|     10:05|    on|
|     10:06|   off|
|     10:07|    on|
|     10:08|    on|
|     10:09|   off|
|     10:11|    on|
|     10:12|   off|
+----------+------+



In [0]:
w=Window.orderBy("event_time")
event_status.withColumn("previous_status",lag("status",1,col("status")).over(w))\
.withColumn("flag"
            ,sum(when((col("status")=='on') & (col("previous_status")=='off'),1).otherwise(0)).over(w)
            ).groupBy(col("flag")).agg(min(col("event_time")).alias("login"),max(col("event_time")).alias("logout"),(count("*")-1).alias("cnt")).show()

w1=Window.orderBy(col("event_time").desc())
event_status.withColumn("flag",sum(when((col("status")=='off'),1) .otherwise(0)).over(w1)).groupBy(col("flag")).agg(min(col("event_time")).alias("login"),max(col("event_time")).alias("logout"),(count("*")-1).alias("cnt")).show()


+----+-----+------+---+
|flag|login|logout|cnt|
+----+-----+------+---+
|   0|10:01| 10:04|  3|
|   1|10:05| 10:06|  1|
|   2|10:07| 10:09|  2|
|   3|10:11| 10:12|  1|
+----+-----+------+---+

+----+-----+------+---+
|flag|login|logout|cnt|
+----+-----+------+---+
|   1|10:11| 10:12|  1|
|   2|10:07| 10:09|  2|
|   3|10:05| 10:06|  1|
|   4|10:01| 10:04|  3|
+----+-----+------+---+



In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Step 2: Define schema
schema = StructType([
    StructField("name", StringType(), True),
    StructField("city", StringType(), True)
])

# Step 3: Define data
data = [
    ("Sachin", "Mumbai"),
    ("Virat", "Delhi"),
    ("Rahul", "Bangalore"),
    ("Rohit", "Mumbai"),
    ("Mayank", "Bangalore")
]

# Step 4: Create DataFrame
players_location = spark.createDataFrame(data, schema)

# Step 5: Show data
players_location.show()


+------+---------+
|  name|     city|
+------+---------+
|Sachin|   Mumbai|
| Virat|    Delhi|
| Rahul|Bangalore|
| Rohit|   Mumbai|
|Mayank|Bangalore|
+------+---------+



In [0]:
window_spec = Window.partitionBy("city").orderBy("name")
df_with_row = players_location.withColumn("row_num", row_number().over(window_spec))

# Pivot the table
pivoted = df_with_row.groupBy(col("row_num")).pivot("city").agg(first("name"))

# Optional: Rename columns to uppercase
for col_name in pivoted.columns:
    pivoted = pivoted.withColumnRenamed(col_name, col_name.upper())

# Show final result
pivoted.select("BANGALORE", "MUMBAI", "DELHI").show()

+---------+------+-----+
|BANGALORE|MUMBAI|DELHI|
+---------+------+-----+
|   Mayank| Rohit|Virat|
|    Rahul|Sachin| NULL|
+---------+------+-----+



In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import first, col

spark = SparkSession.builder.appName("FirstAggregateExample").getOrCreate()

data = [("A", 1), ("B", 2), ("A", 3), ("B", 4)]
columns = ["group_key", "value"]
df = spark.createDataFrame(data, columns)

# Get the first 'value' for each 'group_key'
result_df = df.groupBy("group_key").agg(first(col("value")).alias("first_value"))
result_df.show()

# Get the first non-null 'value' for each 'group_key'
data_with_nulls = [("X", None), ("Y", 10), ("X", 20), ("Y", None)]
df_nulls = spark.createDataFrame(data_with_nulls, columns)
result_with_nulls_df = df_nulls.groupBy("group_key").agg(first(col("value"), ignorenulls=True).alias("first_non_null_value"))
result_with_nulls_df.show()



+---------+-----------+
|group_key|first_value|
+---------+-----------+
|        A|          1|
|        B|          2|
+---------+-----------+

+---------+--------------------+
|group_key|first_non_null_value|
+---------+--------------------+
|        X|                  20|
|        Y|                  10|
+---------+--------------------+



In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("company", StringType(), True),
    StructField("salary", IntegerType(), True)
])

# Data rows
data = [
    (1, 'A', 2341), (2, 'A', 341), (3, 'A', 15), (4, 'A', 15314), (5, 'A', 451), (6, 'A', 513),
    (7, 'B', 15), (8, 'B', 13), (9, 'B', 1154), (10, 'B', 1345), (11, 'B', 1221), (12, 'B', 234),
    (13, 'C', 2345), (14, 'C', 2645), (15, 'C', 2645), (16, 'C', 2652), (17, 'C', 65)
]

# Create DataFrame
employee_df = spark.createDataFrame(data, schema)

# Show DataFrame
employee_df.show()

+------+-------+------+
|emp_id|company|salary|
+------+-------+------+
|     1|      A|  2341|
|     2|      A|   341|
|     3|      A|    15|
|     4|      A| 15314|
|     5|      A|   451|
|     6|      A|   513|
|     7|      B|    15|
|     8|      B|    13|
|     9|      B|  1154|
|    10|      B|  1345|
|    11|      B|  1221|
|    12|      B|   234|
|    13|      C|  2345|
|    14|      C|  2645|
|    15|      C|  2645|
|    16|      C|  2652|
|    17|      C|    65|
+------+-------+------+



In [0]:
w=Window.partitionBy(col("company")).orderBy("salary")
w1=Window.partitionBy(col("company"))
employee_df.withColumn("rnk",row_number().over(w)).withColumn("cnt",count("*").over(w1)).filter(col("rnk").between(col("cnt")/2,col("cnt")/2+1)).groupBy(col("company")).agg(avg(col("salary")).alias("avg_salary")).show()

+-------+----------+
|company|avg_salary|
+-------+----------+
|      A|     482.0|
|      B|     694.0|
|      C|    2645.0|
+-------+----------+



In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

data = [
    (1, "2017-07-01", 10),
    (2, "2017-07-02", 109),
    (3, "2017-07-03", 150),
    (4, "2017-07-04", 99),
    (5, "2017-07-05", 145),
    (6, "2017-07-06", 1455),
    (7, "2017-07-07", 199),
    (8, "2017-07-08", 188)
]

# Create DataFrame with visit_date as string first
df_raw = spark.createDataFrame(data, ["id", "visit_date", "no_of_people"])

# Convert visit_date column from string to date type
stadium_df = df_raw.withColumn("visit_date", to_date("visit_date", "yyyy-MM-dd"))

stadium_df.show()


+---+----------+------------+
| id|visit_date|no_of_people|
+---+----------+------------+
|  1|2017-07-01|          10|
|  2|2017-07-02|         109|
|  3|2017-07-03|         150|
|  4|2017-07-04|          99|
|  5|2017-07-05|         145|
|  6|2017-07-06|        1455|
|  7|2017-07-07|         199|
|  8|2017-07-08|         188|
+---+----------+------------+



In [0]:
w=Window.orderBy("id")
stadium_df.filter(col("no_of_people")>100).withColumn("flag",dateadd(col("visit_date"),-1*row_number().over(w))).withColumn("cnt",count("*").over(Window.partitionBy(col("flag")))).filter(col("cnt")>3).drop("cnt","flag").show()

/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---+----------+------------+
| id|visit_date|no_of_people|
+---+----------+------------+
|  5|2017-07-05|         145|
|  6|2017-07-06|        1455|
|  7|2017-07-07|         199|
|  8|2017-07-08|         188|
+---+----------+------------+



In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

data = [
    (1, "Alice", 20),
    (2, None, 25),
    (3, "Bob", None),
    (4, None, None)
]

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", StringType(), True)
])

df = spark.createDataFrame(data, schema)
print("Original DataFrame:")
df.show()
# Fill all string nulls with "Unknown" and all integer nulls with 0
filled_df = df.fillna({"name": "Unknown", "age": 0})
filled_df.show()
filled_df1 = df.fillna('0',subset=['name','age']).show()


Original DataFrame:
+---+-----+----+
| id| name| age|
+---+-----+----+
|  1|Alice|  20|
|  2| NULL|  25|
|  3|  Bob|NULL|
|  4| NULL|NULL|
+---+-----+----+

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|  Alice| 20|
|  2|Unknown| 25|
|  3|    Bob|  0|
|  4|Unknown|  0|
+---+-------+---+

+---+-----+---+
| id| name|age|
+---+-----+---+
|  1|Alice| 20|
|  2|    0| 25|
|  3|  Bob|  0|
|  4|    0|  0|
+---+-----+---+



In [0]:

data = [
    (1, None, "Alice"),
    (2, "Bob", None),
    (3, None, None)
]

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name1", StringType(), True),
    StructField("name2", StringType(), True)
])

df = spark.createDataFrame(data, schema)
df.show()
df_with_coalesce = df.withColumn("final_name", coalesce(df["name1"], df["name2"]))
df_with_coalesce.show()


+---+-----+-----+
| id|name1|name2|
+---+-----+-----+
|  1| NULL|Alice|
|  2|  Bob| NULL|
|  3| NULL| NULL|
+---+-----+-----+

+---+-----+-----+----------+
| id|name1|name2|final_name|
+---+-----+-----+----------+
|  1| NULL|Alice|     Alice|
|  2|  Bob| NULL|       Bob|
|  3| NULL| NULL|      NULL|
+---+-----+-----+----------+



In [0]:
data = [
    (1, "Alice", "HR"),
    (2, "Bob", "IT"),
    (3, "Alice", "HR"),     # duplicate on name & department
    (4, None, "Finance"),   # null in name
    (5, "Eve", None),       # null in department
    (6, None, None)        # duplicate on name & department
]

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("department", StringType(), True)
])

df = spark.createDataFrame(data, schema)

print("Original Data:")
df.show()
# Drop rows where 'name' or 'department' is null
df_no_nulls = df.dropna(subset=["name", "department"])

print("After dropna(subset=['name', 'department']):")
df_no_nulls.show()
# Drop duplicates based only on 'name' and 'department'
df_unique = df_no_nulls.dropDuplicates(subset=["name", "department"])

print("After dropDuplicates(subset=['name', 'department']):")
df_unique.show()


Original Data:
+---+-----+----------+
| id| name|department|
+---+-----+----------+
|  1|Alice|        HR|
|  2|  Bob|        IT|
|  3|Alice|        HR|
|  4| NULL|   Finance|
|  5|  Eve|      NULL|
|  6| NULL|      NULL|
+---+-----+----------+

After dropna(subset=['name', 'department']):
+---+-----+----------+
| id| name|department|
+---+-----+----------+
|  1|Alice|        HR|
|  2|  Bob|        IT|
|  3|Alice|        HR|
+---+-----+----------+

After dropDuplicates(subset=['name', 'department']):
+---+-----+----------+
| id| name|department|
+---+-----+----------+
|  1|Alice|        HR|
|  2|  Bob|        IT|
+---+-----+----------+



In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
data = [
    ("2020-01-02", 3),
    ("2020-07-01", 7),
    ("2021-01-01", 3),
    ("2021-02-03", 19),
    ("2022-12-01", 3),
    ("2022-12-15", 3),
    ("2022-02-28", 12)
]

# Define schema (keep date as string for conversion)
schema = StructType([
    StructField("business_date", StringType(), True),
    StructField("city_id", IntegerType(), True)
])

# Create initial DataFrame
df_raw = spark.createDataFrame(data, schema)

# Convert string to actual date type
from pyspark.sql.functions import col
business_city_df = df_raw.withColumn("business_date", to_date(col("business_date"), "yyyy-MM-dd"))

# Show the result
business_city_df.show()


+-------------+-------+
|business_date|city_id|
+-------------+-------+
|   2020-01-02|      3|
|   2020-07-01|      7|
|   2021-01-01|      3|
|   2021-02-03|     19|
|   2022-12-01|      3|
|   2022-12-15|      3|
|   2022-02-28|     12|
+-------------+-------+



In [0]:
business_city_df.groupBy(col("city_id")).agg(min("business_date").alias("min_date")).groupBy(year(col("min_date"))).agg(count("*").alias("cnt")).show()


# Step 1: Create window partitioned by city_id
window_spec = Window.partitionBy("city_id")
# Step 2: Add min_date column
df_with_min = business_city_df.withColumn("min_date", min("business_date").over(window_spec))
# Step 3: Add year and conditional count flag
df_flagged = df_with_min.withColumn("year", year("business_date")) \
                        .withColumn("is_min", when(col("business_date") == col("min_date"), 1).otherwise(None))
# Step 4: Group by year and count
result = df_flagged.groupBy("year").agg(count("is_min").alias("count"))
# Show result
result.show()

+--------------+---+
|year(min_date)|cnt|
+--------------+---+
|          2021|  1|
|          2020|  2|
|          2022|  1|
+--------------+---+

+----+-----+
|year|count|
+----+-----+
|2021|    1|
|2020|    2|
|2022|    1|
+----+-----+



In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

schema = StructType([
    StructField("seat", StringType(), True),
    StructField("occupancy", IntegerType(), True)
])

# Data as list of tuples
data = [
    ('a1',1),('a2',1),('a3',0),('a4',0),('a5',0),('a6',0),('a7',1),('a8',1),('a9',0),('a10',0),
    ('b1',0),('b2',0),('b3',0),('b4',1),('b5',1),('b6',1),('b7',1),('b8',0),('b9',0),('b10',0),
    ('c1',0),('c2',1),('c3',0),('c4',1),('c5',1),('c6',0),('c7',1),('c8',0),('c9',0),('c10',1)
]

# Create DataFrame
movie_df = spark.createDataFrame(data, schema)

# Show DataFrame
movie_df.show(30)


+----+---------+
|seat|occupancy|
+----+---------+
|  a1|        1|
|  a2|        1|
|  a3|        0|
|  a4|        0|
|  a5|        0|
|  a6|        0|
|  a7|        1|
|  a8|        1|
|  a9|        0|
| a10|        0|
|  b1|        0|
|  b2|        0|
|  b3|        0|
|  b4|        1|
|  b5|        1|
|  b6|        1|
|  b7|        1|
|  b8|        0|
|  b9|        0|
| b10|        0|
|  c1|        0|
|  c2|        1|
|  c3|        0|
|  c4|        1|
|  c5|        1|
|  c6|        0|
|  c7|        1|
|  c8|        0|
|  c9|        0|
| c10|        1|
+----+---------+



In [0]:
w=Window.partitionBy(col("row_id")).orderBy(col("seat_no"))
movie_df.withColumn("row_id",substring(col("seat"),1,1)).withColumn("seat_no",substring(col("seat"),2,2).cast("int")).filter(col("occupancy")==0).withColumn("rnk",row_number().over(w)).withColumn("flag",col("seat_no")-col("rnk")).withColumn("count",count("*").over(Window.partitionBy(col("row_id"),col("flag")))).filter(col("count")>=4).select("seat").show()

+----+
|seat|
+----+
|  a3|
|  a4|
|  a5|
|  a6|
+----+



In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

# Define schema
schema = StructType([
    StructField("call_type", StringType(), True),
    StructField("call_number", StringType(), True),
    StructField("call_duration", IntegerType(), True)
])

# Data
data = [
    ('OUT','181868',13), ('OUT','2159010',8), ('OUT','2159010',178),
    ('SMS','4153810',1), ('OUT','2159010',152), ('OUT','9140152',18),
    ('SMS','4162672',1), ('SMS','9168204',1), ('OUT','9168204',576),
    ('INC','2159010',5), ('INC','2159010',4), ('SMS','2159010',1),
    ('SMS','4535614',1), ('OUT','181868',20), ('INC','181868',54),
    ('INC','218748',20), ('INC','2159010',9), ('INC','197432',66),
    ('SMS','2159010',1), ('SMS','4535614',1)
]

# Create DataFrame
call_details_df = spark.createDataFrame(data, schema=schema)
call_details_df.show()

+---------+-----------+-------------+
|call_type|call_number|call_duration|
+---------+-----------+-------------+
|      OUT|     181868|           13|
|      OUT|    2159010|            8|
|      OUT|    2159010|          178|
|      SMS|    4153810|            1|
|      OUT|    2159010|          152|
|      OUT|    9140152|           18|
|      SMS|    4162672|            1|
|      SMS|    9168204|            1|
|      OUT|    9168204|          576|
|      INC|    2159010|            5|
|      INC|    2159010|            4|
|      SMS|    2159010|            1|
|      SMS|    4535614|            1|
|      OUT|     181868|           20|
|      INC|     181868|           54|
|      INC|     218748|           20|
|      INC|    2159010|            9|
|      INC|     197432|           66|
|      SMS|    2159010|            1|
|      SMS|    4535614|            1|
+---------+-----------+-------------+



In [0]:
call_details_df.groupBy(col("call_number")) \
    .agg(sum( 
        when(col("call_type") == "OUT", col("call_duration")).otherwise(None)
        ).alias("total_out_duration"),
         sum( 
        when(col("call_type") == "INC", col("call_duration")).otherwise(None)
        ).alias("total_inc_duration")
         ).filter((col("total_out_duration").isNotNull()) & (col("total_inc_duration").isNotNull()) & (col("total_out_duration") > col("total_inc_duration"))).show()

+-----------+------------------+------------------+
|call_number|total_out_duration|total_inc_duration|
+-----------+------------------+------------------+
|    2159010|               338|                18|
+-----------+------------------+------------------+



In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window


# Define schema
brands_schema = StructType([
    StructField("category", StringType(), True),
    StructField("brand_name", StringType(), True)
])

# Data
brands_data = [
    ('chocolates', '5-star'),
    (None, 'dairy milk'),
    (None, 'perk'),
    (None, 'eclair'),
    ('Biscuits', 'britannia'),
    (None, 'good day'),
    (None, 'boost')
]

# Create DataFrame
brands_df = spark.createDataFrame(brands_data, schema=brands_schema)

# Show DataFrame
brands_df.show()

No such comm: LSP_COMM_ID
No such comm: LSP_COMM_ID


+----------+----------+
|  category|brand_name|
+----------+----------+
|chocolates|    5-star|
|      NULL|dairy milk|
|      NULL|      perk|
|      NULL|    eclair|
|  Biscuits| britannia|
|      NULL|  good day|
|      NULL|     boost|
+----------+----------+



In [0]:
brands_df.withColumn("dummy",lit(1)).withColumn("rnk",row_number().over(Window.orderBy(col("dummy")))).withColumn("last_not_null",last(col("category"), ignorenulls=True).over(Window.orderBy(col("rnk")).rowsBetween(Window.unboundedPreceding, Window.currentRow))).show()

+----------+----------+-----+---+-------------+
|  category|brand_name|dummy|rnk|last_not_null|
+----------+----------+-----+---+-------------+
|chocolates|    5-star|    1|  1|   chocolates|
|      NULL|dairy milk|    1|  2|   chocolates|
|      NULL|      perk|    1|  3|   chocolates|
|      NULL|    eclair|    1|  4|   chocolates|
|  Biscuits| britannia|    1|  5|     Biscuits|
|      NULL|  good day|    1|  6|     Biscuits|
|      NULL|     boost|    1|  7|     Biscuits|
+----------+----------+-----+---+-------------+



In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window


# Start Spark session
spark = SparkSession.builder.appName("StudentsExams").getOrCreate()

# Define schemas
students_schema = StructType([
    StructField("student_id", IntegerType(), True),
    StructField("student_name", StringType(), True)
])

exams_schema = StructType([
    StructField("exam_id", IntegerType(), True),
    StructField("student_id", IntegerType(), True),
    StructField("score", IntegerType(), True)
])

# Define data
students_data = [
    (1, 'Daniel'), (2, 'Jade'), (3, 'Stella'),
    (4, 'Jonathan'), (5, 'Will')
]

exams_data = [
    (10, 1, 70), (10, 2, 80), (10, 3, 90),
    (20, 1, 80),
    (30, 1, 70), (30, 3, 80), (30, 4, 90),
    (40, 1, 60), (40, 2, 70), (40, 4, 80)
]

# Create DataFrames
students_df = spark.createDataFrame(students_data, schema=students_schema)
exams_df = spark.createDataFrame(exams_data, schema=exams_schema)

# Show data
students_df.show()
exams_df.show()


+----------+------------+
|student_id|student_name|
+----------+------------+
|         1|      Daniel|
|         2|        Jade|
|         3|      Stella|
|         4|    Jonathan|
|         5|        Will|
+----------+------------+

+-------+----------+-----+
|exam_id|student_id|score|
+-------+----------+-----+
|     10|         1|   70|
|     10|         2|   80|
|     10|         3|   90|
|     20|         1|   80|
|     30|         1|   70|
|     30|         3|   80|
|     30|         4|   90|
|     40|         1|   60|
|     40|         2|   70|
|     40|         4|   80|
+-------+----------+-----+



In [0]:
exams_df.withColumn("min_score_examid", min(col("score")).over(Window.partitionBy(col("exam_id")))) \
    .withColumn("max_score_examid", max(col("score")).over(Window.partitionBy(col("exam_id"))))\
    .groupBy(col("student_id")) \
    .agg(max(
        (when(
            (col("score")==col("min_score_examid")) | (col("score")==col("max_score_examid")), 1).otherwise(0)
         )).alias("flag")) \
    .filter(col("flag")==0).show()
  

+----------+----+
|student_id|flag|
+----------+----+
|         2|   0|
+----------+----+



In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *

# Start Spark session
# spark = SparkSession.builder.appName("PhoneLog").getOrCreate()

# Define schema
phonelog_schema = StructType([
    StructField("Callerid", IntegerType(), True),
    StructField("Recipientid", IntegerType(), True),
    StructField("Datecalled", StringType(), True)
])

# Define data
phonelog_data = [
    (1, 2, '2019-01-01 09:00:00'),
    (1, 3, '2019-01-01 17:00:00'),
    (1, 4, '2019-01-01 23:00:00'),
    (2, 5, '2019-07-05 09:00:00'),
    (2, 3, '2019-07-05 17:00:00'),
    (2, 3, '2019-07-05 17:20:00'),
    (2, 5, '2019-07-05 23:00:00'),
    (2, 3, '2019-08-01 09:00:00'),
    (2, 3, '2019-08-01 17:00:00'),
    (2, 5, '2019-08-01 19:30:00'),
    (2, 4, '2019-08-02 09:00:00'),
    (2, 5, '2019-08-02 10:00:00'),
    (2, 5, '2019-08-02 10:45:00'),
    (2, 4, '2019-08-02 11:00:00')
]

# Create DataFrame
phonelog_df = spark.createDataFrame(phonelog_data, schema=phonelog_schema)

# Show data
phonelog_df.show(truncate=False)


+--------+-----------+-------------------+
|Callerid|Recipientid|Datecalled         |
+--------+-----------+-------------------+
|1       |2          |2019-01-01 09:00:00|
|1       |3          |2019-01-01 17:00:00|
|1       |4          |2019-01-01 23:00:00|
|2       |5          |2019-07-05 09:00:00|
|2       |3          |2019-07-05 17:00:00|
|2       |3          |2019-07-05 17:20:00|
|2       |5          |2019-07-05 23:00:00|
|2       |3          |2019-08-01 09:00:00|
|2       |3          |2019-08-01 17:00:00|
|2       |5          |2019-08-01 19:30:00|
|2       |4          |2019-08-02 09:00:00|
|2       |5          |2019-08-02 10:00:00|
|2       |5          |2019-08-02 10:45:00|
|2       |4          |2019-08-02 11:00:00|
+--------+-----------+-------------------+



In [0]:
phonelog_df.withColumn("datecalled", to_timestamp(col("Datecalled"), "yyyy-MM-dd HH:mm:ss")).withColumn("date_call", col("datecalled").cast("date")).withColumn("first_receipient",
    first(col("Recipientid")).over(
        Window.partitionBy(col("callerid"),col("date_call")).orderBy(col("datecalled")))).withColumn("last_receipient",
    first(col("Recipientid")).over(
        Window.partitionBy(col("callerid"),col("date_call")).orderBy(col("datecalled").desc()))) \
        .withColumn("flag",when(col("first_receipient")==col("last_receipient"),0).otherwise(1)).groupBy(col("Callerid"),col("date_call")).agg(max("flag").alias("flag")).filter(col("flag")==0).show(truncate=False)

+--------+----------+----+
|Callerid|date_call |flag|
+--------+----------+----+
|2       |2019-07-05|0   |
|2       |2019-08-02|0   |
+--------+----------+----+



In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as _sum,avg
from pyspark.sql.window import Window

# Start Spark session
spark = SparkSession.builder.appName("CandidateSelection").getOrCreate()

# Create the candidates data
data = [
    (1, 'Junior', 10000),
    (2, 'Junior', 15000),
    (3, 'Junior', 40000),
    (4, 'Senior', 16000),
    (5, 'Senior', 20000),
    (6, 'Senior', 50000)
]

# Create DataFrame
columns = ["emp_id", "experience", "salary"]
df = spark.createDataFrame(data, columns)

# Define window for running salary
window_spec = Window.partitionBy("experience").orderBy(col("experience").desc(), col("salary").asc())

# Add running salary column
df_with_running_sal = df.withColumn("running_sal", _sum("salary").over(window_spec))

# Filter senior candidates with running salary <= 70000
senior_hired_df = df_with_running_sal.filter((col("experience") == "Senior") & (col("running_sal") <= 70000))

# Calculate remaining budget for juniors
remaining_budget = senior_hired_df.select(_sum("running_sal").alias("remaining_sal")).collect()[0]["remaining_sal"]
print("Remaining budget for juniors:", remaining_budget)
remaining_budget = 70000 - remaining_budget

# Filter junior candidates with running salary <= remaining budget
junior_hired_df = df_with_running_sal.filter((col("experience") == "Junior") & (col("running_sal") <= remaining_budget))

# Combine both results and order by emp_id
final_df = senior_hired_df.union(junior_hired_df).orderBy("emp_id")

# Show the final result
final_df.show()



Remaining budget for juniors: 52000
+------+----------+------+-----------+
|emp_id|experience|salary|running_sal|
+------+----------+------+-----------+
|     1|    Junior| 10000|      10000|
|     4|    Senior| 16000|      16000|
|     5|    Senior| 20000|      36000|
+------+----------+------+-----------+



In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *


# Start Spark session
spark = SparkSession.builder.appName("EmployeeHierarchy").getOrCreate()

# Define schema
emp_schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("emp_name", StringType(), True),
    StructField("department_id", IntegerType(), True),
    StructField("salary", IntegerType(), True),
    StructField("manager_id", IntegerType(), True),
    StructField("emp_age", IntegerType(), True)
])

# Define data
emp_data = [
    (1, 'Ankit', 100, 10000, 4, 39),
    (2, 'Mohit', 100, 15000, 5, 48),
    (3, 'Vikas', 100, 12000, 4, 37),
    (4, 'Rohit', 100, 14000, 2, 16),
    (5, 'Mudit', 200, 20000, 6, 55),
    (6, 'Agam', 200, 12000, 2, 14),
    (7, 'Sanjay', 200, 9000, 2, 13),
    (8, 'Ashish', 200, 5000, 2, 12),
    (9, 'Mukesh', 300, 6000, 6, 51),
    (10, 'Rakesh', 500, 7000, 6, 50)
]

# Create DataFrame
emp_df = spark.createDataFrame(emp_data, schema=emp_schema)

# Create manager_info by joining emp with itself to get manager name
manager_info_df = emp_df.alias("e").join(
    emp_df.alias("m"),
    col("e.manager_id") == col("m.emp_id"),
    how="left"
).select(
    "e.emp_id",
    "e.emp_name","e.salary",
    "e.manager_id",
    col("m.emp_name").alias("manager_name"),
    col("m.manager_id").alias("s_manager_id"),
    col("m.salary").alias("manager_salary")
)

# Join again to get senior manager name
final_df = manager_info_df.alias("m").join(
    manager_info_df.alias("sm"),
    col("m.s_manager_id") == col("sm.emp_id"),"inner"
).select(
    "m.emp_id",
    "m.emp_name",
    "m.salary",
    "m.manager_id",
    "m.manager_name",
    "m.manager_salary",
    col("sm.s_manager_id").alias("senior_manager_id"),
    col("sm.emp_name").alias("senior_manager_name"),
    col("sm.salary").alias("senior_manager_salary")

).orderBy("m.emp_id")

# Show the result
final_df.filter(col("salary")>=col("senior_manager_salary")).show(truncate=False)



+------+--------+------+----------+------------+--------------+-----------------+-------------------+---------------------+
|emp_id|emp_name|salary|manager_id|manager_name|manager_salary|senior_manager_id|senior_manager_name|senior_manager_salary|
+------+--------+------+----------+------------+--------------+-----------------+-------------------+---------------------+
|2     |Mohit   |15000 |5         |Mudit       |20000         |5                |Agam               |12000                |
|5     |Mudit   |20000 |6         |Agam        |12000         |6                |Mohit              |15000                |
+------+--------+------+----------+------------+--------------+-----------------+-------------------+---------------------+



In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *


# Create Spark session
spark = SparkSession.builder.appName("HotelAmenities").getOrCreate()

# Sample data
data = [
    (123, 'Pool'),
    (123, 'Kitchen'),
    (234, 'Hot Tub'),
    (234, 'Fireplace'),
    (345, 'Kitchen'),
    (345, 'Pool'),
    (456, 'Pool')
]

# Define schema
columns = ["rental_id", "amenities"]

# Create DataFrame
hotel_df = spark.createDataFrame(data, columns)

# Show the DataFrame
hotel_df.show()


+---------+---------+
|rental_id|amenities|
+---------+---------+
|      123|     Pool|
|      123|  Kitchen|
|      234|  Hot Tub|
|      234|Fireplace|
|      345|  Kitchen|
|      345|     Pool|
|      456|     Pool|
+---------+---------+



In [0]:
hotel_join = hotel_df.groupBy("rental_id").agg(
    array_join(sort_array(collect_list("amenities")),  ',').alias("amenities")
 )

hotel_join.alias("a").join(
    hotel_join.alias("b"),
    (col("a.amenities") == col("b.amenities")) & (col("a.rental_id") < col("b.rental_id")),
    "inner"
).agg(count("*").alias("total_combination")).show()

+-----------------+
|total_combination|
+-----------------+
|                1|
+-----------------+



In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum
from pyspark.sql.window import Window

# Create Spark session
spark = SparkSession.builder.appName("GlobalSum").getOrCreate()

# Sample data
data = [("A", 10), ("B", 20), ("C", 30)]
df = spark.createDataFrame(data, ["category", "value"])

# Define a global window (no partition, no order, no frame)
windowSpec = Window.orderBy()

# Apply global sum
df_with_sum = df.withColumn("global_sum", sum("value").over(windowSpec))

df_with_sum.show()


/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+--------+-----+----------+
|category|value|global_sum|
+--------+-----+----------+
|       A|   10|        60|
|       B|   20|        60|
|       C|   30|        60|
+--------+-----+----------+

